In [ ]:
import os
import random
import itertools
import json
import numpy as np
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter

import torchvision
from torchvision import transforms, models
from PIL import Image

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
from sklearn.model_selection import StratifiedKFold
import torch.nn.functional as F

In [ ]:
# Device setup 
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
    
print(f'Using device: {device}')


In [ ]:
def seed_everything(seed: int):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False  

seed_everything(7)


In [ ]:
# IMAGE TRANSFORMS 
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),                  
    transforms.ColorJitter(brightness=0.2, contrast=0.2),       
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std)
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std)
])

print('Transforms ready.')


In [ ]:
# STRONG AUGMENTATION OPTION

strong_train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.1),
    transforms.RandomGrayscale(p=0.1),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.1)),
])
print('Strong augmentation transform ready.')


In [ ]:
# EXPERIMENT CONFIGURATION 

MAX_PER_CLASS   = None       
FREEZE_BACKBONE = False      # True -> only train final classification head
USE_STRONG_AUG  = False      # True -> GaussianBlur + RandomGrayscale + RandomErasing
ARCH            = 'resnet18' 
TEST_SPLIT      = 0.30       

print(f'ARCH={ARCH}  |  MAX_PER_CLASS={MAX_PER_CLASS}  |  FREEZE={FREEZE_BACKBONE}')
print(f'STRONG_AUG={USE_STRONG_AUG}  |  TEST_SPLIT={TEST_SPLIT}')


In [ ]:
# DATA LOADING 
BASE_PATH = os.path.expanduser('~/Desktop/george/rocks_spectral_224')

folders_183 = {
    'S10Granite':           os.path.join(BASE_PATH, 'S10Granite_1-83Hz_Spectral'),
    'Holstein_Sandstone':   os.path.join(BASE_PATH, 'Holstein_Sandstone_1-83Hz_Spectral'),
    'Leitendorf_Limestone': os.path.join(BASE_PATH, 'Leitendorf_Limestone_1-83Hz_Spectral'),
}

folders_510 = {
    'S10Granite':           os.path.join(BASE_PATH, 'S10Granite_5-10Hz_Spectral'),
    'Holstein_Sandstone':   os.path.join(BASE_PATH, 'Holstein_Sandstone_5-10Hz_Spectral'),
    'Leitendorf_Limestone': os.path.join(BASE_PATH, 'Leitendorf_Limestone_5-10Hz_Spectral'),
}

VALID_EXTENSIONS = ('.jpg', '.jpeg', '.bmp', '.png')

def collect_image_paths(folders_dict, max_per_class=None):
    image_paths = []
    labels      = []
    class_names = list(folders_dict.keys())

    for label, (rock_name, folder) in enumerate(folders_dict.items()):
        files = [f for f in os.listdir(folder) if f.lower().endswith(VALID_EXTENSIONS)]
        if max_per_class is not None:
            files = random.sample(files, min(max_per_class, len(files)))
        for fname in files:
            image_paths.append(os.path.join(folder, fname))
            labels.append(label)
        print(f'{rock_name}: {len(files)} images')

    return image_paths, labels, class_names


print('Loading 1.83 Hz image paths')
paths_183, labels_183, class_names = collect_image_paths(folders_183, max_per_class=MAX_PER_CLASS)
print(f'Total: {len(paths_183)} images\n')

print('Loading 5.10 Hz image paths')
paths_510, labels_510, _ = collect_image_paths(folders_510, max_per_class=MAX_PER_CLASS)
print(f'Total: {len(paths_510)} images')


In [ ]:
class_names

In [ ]:
# VISUALIZE SAMPLE IMAGES 
for speed_tag, paths, _labels in [
    ('1.83 Hz', paths_183, labels_183),
    ('5.10 Hz', paths_510, labels_510)
]:
    fig, axes = plt.subplots(3, 3, figsize=(12, 10))

    for i, rock_name in enumerate(class_names):
        rock_indices = [j for j, l in enumerate(_labels) if l == i][:3]
        for k, idx in enumerate(rock_indices):
            img = Image.open(paths[idx]).convert('RGB')
            axes[i][k].imshow(img)
            axes[i][k].set_title(f'{rock_name}' if k == 1 else '')
            axes[i][k].axis('off')

    plt.suptitle(f'Sample Spectral Images - {speed_tag}', fontsize=14)
    plt.tight_layout()
    _tag8 = speed_tag.replace(' ', '').replace('.', '')
    _sp8  = os.path.join('results_resnet_full_evaluation', f'sample_images_{_tag8}.png')
    os.makedirs('results_resnet_full_evaluation', exist_ok=True)
    plt.savefig(_sp8, dpi=150, bbox_inches='tight')
    print(f'[SAVED] Sample images -> {_sp8}')
    plt.show()

In [ ]:
# TRAIN/TEST SPLIT 
(
    paths_train_183, paths_test_183,
    labels_train_183, labels_test_183
) = train_test_split(
    paths_183, labels_183,
    test_size=TEST_SPLIT, random_state=3, stratify=labels_183
)

(
    paths_train_510, paths_test_510,
    labels_train_510, labels_test_510
) = train_test_split(
    paths_510, labels_510,
    test_size=TEST_SPLIT, random_state=3, stratify=labels_510
)

print('1.83 Hz -> Train:', len(paths_train_183), ' Test:', len(paths_test_183))
print('5.10 Hz -> Train:', len(paths_train_510), ' Test:', len(paths_test_510))

In [ ]:
# DATASET CLASS 
class SpectralImageDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels      = labels
        self.transform   = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img   = Image.open(self.image_paths[idx]).convert('RGB')
        label = self.labels[idx]
        if self.transform:
            img = self.transform(img)
        return img, label


_active_train_transform = strong_train_transform if USE_STRONG_AUG else train_transform
print(f'Using {"STRONG" if USE_STRONG_AUG else "standard"} augmentation for training.')

train_dataset_183 = SpectralImageDataset(paths_train_183, labels_train_183, transform=_active_train_transform)
test_dataset_183  = SpectralImageDataset(paths_test_183,  labels_test_183,  transform=test_transform)

train_dataset_510 = SpectralImageDataset(paths_train_510, labels_train_510, transform=_active_train_transform)
test_dataset_510  = SpectralImageDataset(paths_test_510,  labels_test_510,  transform=test_transform)

print('Datasets ready.')

In [ ]:
# HYPERPARAMETERS 
batch_sizes   = [64]
lrs           = [0.0001, 0.00005]
epochs_list   = [20]
weight_decays = [0., 1e-4]

hyperparameters_list = []
for batch_size, lr, epochs, weight_decay in itertools.product(batch_sizes, lrs, epochs_list, weight_decays):
    hyperparameters_list.append({
        'batch_size'  : batch_size,
        'lr'          : lr,
        'epochs'      : epochs,
        'weight_decay': weight_decay
    })

print(f'Generated {len(hyperparameters_list)} combinations.')

In [ ]:
# MODEL BUILDER — ResNet-18 / 34 / 50 and EfficientNet-B0

def build_model(n_classes, arch='resnet18', freeze_backbone=False):
    """Build a pretrained model with a custom classification head."""
    if arch == 'resnet18':
        model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        model.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(model.fc.in_features, n_classes))
    elif arch == 'resnet34':
        model = models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1)
        model.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(model.fc.in_features, n_classes))
    elif arch == 'resnet50':
        model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        model.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(model.fc.in_features, n_classes))
    elif arch == 'efficientnet_b0':
        model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        model.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(model.classifier[1].in_features, n_classes)
        )
    else:
        raise ValueError(f'Unknown arch: {arch}')

    if freeze_backbone:
        head = 'fc' if 'resnet' in arch else 'classifier'
        for name, param in model.named_parameters():
            if head not in name:
                param.requires_grad = False
    return model


def build_resnet18(n_classes, freeze_backbone=False):
    """Backward-compatible alias."""
    return build_model(n_classes, arch='resnet18', freeze_backbone=freeze_backbone)


# Quick test with current config
_m   = build_model(n_classes=3, arch=ARCH, freeze_backbone=FREEZE_BACKBONE)
tot  = sum(p.numel() for p in _m.parameters())
trn  = sum(p.numel() for p in _m.parameters() if p.requires_grad)
print(f'Architecture : {ARCH}')
print(f'Total params : {tot:,}')
print(f'Trainable    : {trn:,}  (frozen={FREEZE_BACKBONE})')
del _m


In [ ]:
# LEARNING RATE FINDER 
# Trains for 1 epoch while linearly increasing LR from lr_start to lr_end then plots loss vs LR (the optimal LR is just before the loss starts rising sharply).

def lr_finder(train_dataset, n_classes, lr_start=1e-7, lr_end=1.0,
              num_iter=200, arch='resnet18'):
    loader  = DataLoader(train_dataset, batch_size=64, shuffle=True,
                         num_workers=4, pin_memory=True, persistent_workers=True)
    model   = build_model(n_classes, arch=arch).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr_start)
    criterion = nn.CrossEntropyLoss()
    scaler    = torch.amp.GradScaler(device.type if device.type == 'cuda' else 'cpu')

    mult   = (lr_end / lr_start) ** (1 / num_iter)
    lr     = lr_start
    best_loss = float('inf')
    avg_loss  = 0.0
    beta      = 0.98   # smoothing

    lrs, losses = [], []
    it = 0

    model.train()
    data_iter = iter(loader)
    while it < num_iter:
        try:
            Xb, yb = next(data_iter)
        except StopIteration:
            data_iter = iter(loader)
            Xb, yb   = next(data_iter)

        Xb, yb = Xb.to(device), yb.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast(device.type):
            out  = model(Xb)
            loss = criterion(out, yb)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        avg_loss = beta * avg_loss + (1 - beta) * loss.item()
        smoothed = avg_loss / (1 - beta ** (it + 1))

        if smoothed < best_loss:
            best_loss = smoothed
        if smoothed > 4 * best_loss:
            print(f'  Early stop at iter {it}')
            break

        lrs.append(lr)
        losses.append(smoothed)

        lr *= mult
        for pg in optimizer.param_groups:
            pg['lr'] = lr
        it += 1

    del model, optimizer, criterion, scaler, loader
    torch.cuda.empty_cache()

    fig_lr, ax_lr = plt.subplots(figsize=(8, 4))
    ax_lr.plot(lrs, losses, color='steelblue')
    ax_lr.set_xscale('log')
    ax_lr.set_xlabel('Learning Rate (log scale)')
    ax_lr.set_ylabel('Smoothed Loss')
    ax_lr.set_title(f'LR Finder — 1.83 Hz  ({arch})')
    ax_lr.grid(True, alpha=0.3)
    plt.tight_layout()
    os.makedirs('results_resnet_full_evaluation', exist_ok=True)
    lr_path = 'results_resnet_full_evaluation/lr_finder.png'
    plt.savefig(lr_path, dpi=150, bbox_inches='tight')
    print(f'[SAVED] LR finder -> {lr_path}')
    plt.show()
    return lrs, losses


lr_finder_lrs, lr_finder_losses = lr_finder(
    train_dataset_183, n_classes=3, arch=ARCH)


In [ ]:
# TRAINING FUNCTION 
def run_experiments(train_dataset, test_dataset, speed_tag, results_dir, n_classes):

    experiment_results = []
    os.makedirs(results_dir, exist_ok=True)

    for idx, config in enumerate(hyperparameters_list):
        print(f"\n{'='*20}\nExperiment {idx+1}\nConfig: {config}\n{'='*20}")

        train_loader = DataLoader(
            train_dataset, batch_size=config['batch_size'],
            shuffle=True, num_workers=4, pin_memory=True, persistent_workers=True
        )
        test_loader = DataLoader(
            test_dataset, batch_size=config['batch_size'],
            shuffle=False, num_workers=4, pin_memory=True, persistent_workers=True
        )

        model     = build_model(n_classes, arch=ARCH, freeze_backbone=FREEZE_BACKBONE).to(device)
        optimizer = torch.optim.Adam(
            params=model.parameters(),
            lr=config['lr'],
            weight_decay=config['weight_decay']
        )
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='max', factor=0.5, patience=2
        )
        warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
            optimizer, start_factor=0.1, end_factor=1.0, total_iters=2
        )
        criterion = nn.CrossEntropyLoss()
        scaler = torch.amp.GradScaler(device.type if device.type == 'cuda' else 'cpu')

        best_accuracy   = 0.0
        best_model_path = os.path.join(results_dir, f'best_model_exp_{idx+1}.pth')
        writer          = SummaryWriter(log_dir=os.path.join(results_dir, f'tb_exp{idx+1}_lr{config["lr"]}_wd{config["weight_decay"]}'))

        training_loss, training_acc = [], []
        testing_loss,  testing_acc  = [], []

        for epoch in range(config['epochs']):
            print(f'Epoch: {epoch+1}/{config["epochs"]}')

            # Train
            model.train()
            avg_train_loss, avg_train_acc = [], []
            for X_batch, y_batch in tqdm(train_loader, leave=False):
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)
                optimizer.zero_grad()
                with torch.amp.autocast(device.type):
                    outputs = model(X_batch)
                    loss    = criterion(outputs, y_batch)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
                scaler.step(optimizer)
                scaler.update()
                _, predicted = torch.max(outputs, 1)
                avg_train_loss.append(loss.item())
                avg_train_acc.append((predicted == y_batch).sum().item() / y_batch.size(0))

            avg_acc_train  = sum(avg_train_acc)  / len(avg_train_acc)
            avg_loss_train = sum(avg_train_loss) / len(avg_train_loss)
            training_acc.append(avg_acc_train)
            training_loss.append(avg_loss_train)
            print(f'Training: Loss={avg_loss_train:.4f}, Acc={avg_acc_train:.4f}')

            # Test
            model.eval()
            avg_test_loss, avg_test_acc = [], []
            all_preds, all_true = [], []
            with torch.no_grad():
                for X_batch, y_batch in tqdm(test_loader, leave=False):
                    X_batch = X_batch.to(device)
                    y_batch = y_batch.to(device)
                    outputs = model(X_batch)
                    loss    = criterion(outputs, y_batch)
                    preds   = torch.max(outputs, 1)[1]
                    avg_test_loss.append(loss.item())
                    avg_test_acc.append((preds == y_batch).sum().item() / y_batch.size(0))
                    all_preds.extend(preds.cpu().numpy())
                    all_true.extend(y_batch.cpu().numpy())

            avg_acc_test  = sum(avg_test_acc)  / len(avg_test_acc)
            avg_loss_test = sum(avg_test_loss) / len(avg_test_loss)
            testing_acc.append(avg_acc_test)
            testing_loss.append(avg_loss_test)
            print(f'Testing:  Loss={avg_loss_test:.4f}, Acc={avg_acc_test:.4f}\n')

            if epoch < 2:
                warmup_scheduler.step()
            else:
                scheduler.step(avg_acc_test)

            writer.add_scalars('Loss',     {'train': avg_loss_train, 'test': avg_loss_test}, epoch)
            writer.add_scalars('Accuracy', {'train': avg_acc_train,  'test': avg_acc_test},  epoch)

            if avg_acc_test > best_accuracy:
                best_accuracy = avg_acc_test
                torch.save(model.state_dict(), best_model_path)
                print(f' New Best Acc: {best_accuracy:.4f} - Saved.')
            print('-' * 30)

        # Confusion Matrix using best checkpoint 
        model.load_state_dict(torch.load(best_model_path, map_location=device, weights_only=True))
        model.to(device).eval()
        all_preds, all_true = [], []
        with torch.no_grad():
            for X_batch, y_batch in tqdm(test_loader, leave=False):
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)
                preds   = torch.max(model(X_batch), 1)[1]
                all_preds.extend(preds.cpu().numpy())
                all_true.extend(y_batch.cpu().numpy())

        cm = confusion_matrix(all_true, all_preds)
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=class_names, yticklabels=class_names)
        plt.title(f'Confusion Matrix - {speed_tag} - Exp {idx+1}')
        plt.ylabel('Actual')
        plt.xlabel('Predicted')
        plt.tight_layout()
        cm_path = os.path.join(results_dir, f'confusion_matrix_exp_{idx+1}.png')
        plt.savefig(cm_path, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'[SAVED] Confusion matrix -> {cm_path}')

        report     = classification_report(all_true, all_preds, target_names=class_names,
                                           digits=4, output_dict=True)
        report_str = classification_report(all_true, all_preds, target_names=class_names, digits=4)
        print(f'\nClassification Report — {speed_tag} Exp {idx+1}:\n{report_str}')
        rpt_path = os.path.join(results_dir, f'classification_report_exp_{idx+1}.txt')
        with open(rpt_path, 'w') as _f:
            _f.write(f'Config: {config}\n\n' + report_str)
        print(f'[SAVED] Classification report -> {rpt_path}')

        writer.close()
        del model, optimizer, scheduler, warmup_scheduler, criterion, scaler, train_loader, test_loader
        torch.cuda.empty_cache()  
        experiment_results.append({
            'config'          : config,
            'best_test_acc'   : best_accuracy,
            'final_test_acc'  : testing_acc[-1],
            'final_test_loss' : testing_loss[-1],
            'best_model_path' : best_model_path,
            'class_report'    : report,
            'history': {
                'train_loss': training_loss, 'train_acc': training_acc,
                'test_loss' : testing_loss,  'test_acc' : testing_acc
            }
        })

    print('\n' + '='*40)
    print(f'RESULTS - {speed_tag}')
    print('='*40)
    for res in experiment_results:
        print(f"Config: {res['config']} -> Best Acc: {res['best_test_acc']:.4f}  Final Acc: {res['final_test_acc']:.4f}")

    return experiment_results


# RUN TRAINING 
print('\n' + '#'*50)
print('RESNET-18 TRAINING - 1.83 Hz')
print('#'*50)
results_183 = run_experiments(
    train_dataset_183, test_dataset_183,
    '1.83 Hz', 'results_resnet_full_evaluation/resnet18_results_183', n_classes=3
)

print('\n' + '#'*50)
print('RESNET-18 TRAINING - 5.10 Hz')
print('#'*50)
results_510 = run_experiments(
    train_dataset_510, test_dataset_510,
    '5.10 Hz', 'results_resnet_full_evaluation/resnet18_results_510', n_classes=3
)

In [ ]:
# ANALYZE ALL RESULTS
best_183 = max(results_183, key=lambda x: x['best_test_acc'])
best_510 = max(results_510, key=lambda x: x['best_test_acc'])

for speed_tag, results in [('1.83 Hz', results_183), ('5.10 Hz', results_510)]:
    print('=' * 70)
    print(f'RESNET-18 RESULTS — {speed_tag}')
    print('=' * 70)
    print(f"{'Exp':<5} {'Batch':<7} {'LR':<10} {'WD':<8} {'Best Acc':>10} {'Final Acc':>10} {'Final Loss':>11}")
    print('-' * 70)
    for i, res in enumerate(results):
        cfg = res['config']
        print(f"{i+1:<5} {cfg['batch_size']:<7} {cfg['lr']:<10} {cfg['weight_decay']:<8} "
              f"{res['best_test_acc']*100:>9.2f}% {res['final_test_acc']*100:>9.2f}% {res['final_test_loss']:>11.4f}")
    print()

# Plot ALL training curves
for speed_tag, results in [('1.83 Hz', results_183), ('5.10 Hz', results_510)]:
    n = len(results)
    fig, axes = plt.subplots(n, 2, figsize=(14, 4 * n), squeeze=False)
    fig.suptitle(f'ResNet-18 Training Curves — {speed_tag}', fontsize=14, fontweight='bold')

    for i, res in enumerate(results):
        cfg = res['config']
        label = f"Exp {i+1} | bs={cfg['batch_size']} lr={cfg['lr']} wd={cfg['weight_decay']}"

        axes[i][0].plot(res['history']['train_loss'], color='blue', label='Train')
        axes[i][0].plot(res['history']['test_loss'],  color='red',  label='Test')
        axes[i][0].set_title(f'Loss — {label}')
        axes[i][0].set_xlabel('Epoch')
        axes[i][0].set_ylabel('Loss')
        axes[i][0].legend()
        axes[i][0].grid(True, alpha=0.3)

        axes[i][1].plot(res['history']['train_acc'], color='blue', label='Train')
        axes[i][1].plot(res['history']['test_acc'],  color='red',  label='Test')
        axes[i][1].set_title(f'Accuracy — {label} | Best: {res["best_test_acc"]*100:.2f}%')
        axes[i][1].set_xlabel('Epoch')
        axes[i][1].set_ylabel('Accuracy')
        axes[i][1].legend()
        axes[i][1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.subplots_adjust(top=0.95)
    _dir = 'results_resnet_full_evaluation/resnet18_results_183' if '1.83' in speed_tag else 'results_resnet_full_evaluation/resnet18_results_510'
    _curves_path = os.path.join(_dir, 'training_curves_all.png')
    os.makedirs(_dir, exist_ok=True)
    plt.savefig(_curves_path, dpi=150, bbox_inches='tight')
    print(f'[SAVED] Training curves -> {_curves_path}')
    plt.show()

In [ ]:
# CONFUSION MATRICES - ALL EXPERIMENTS
for speed_tag, results, test_ds in [
    ('1.83Hz', results_183, test_dataset_183),
    ('5.10Hz', results_510, test_dataset_510)
]:
    for res in results:
        cfg = res['config']
        label = f"bs={cfg['batch_size']} lr={cfg['lr']} wd={cfg['weight_decay']}"

        y_true, y_pred = [], []
        loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=4, pin_memory=True, persistent_workers=True)
        model  = build_model(n_classes=3, arch=ARCH).to(device)
        model.load_state_dict(torch.load(res['best_model_path'], map_location=device, weights_only=True))
        model.eval()

        with torch.no_grad():
            for X_batch, y_batch in loader:
                outputs      = model(X_batch.to(device))
                _, predicted = torch.max(outputs, 1)
                y_true.extend(y_batch.tolist())
                y_pred.extend(predicted.cpu().tolist())

        cm   = confusion_matrix(y_true, y_pred)
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
        fig, ax = plt.subplots(figsize=(8, 8))
        disp.plot(ax=ax, cmap=plt.cm.Blues, xticks_rotation='vertical')
        plt.title(f'ResNet-18 Confusion Matrix - {speed_tag} - {label}\nBest Acc: {res["best_test_acc"]*100:.2f}%')
        _results_dir = os.path.dirname(res['best_model_path'])
        _cm_fname = f'confusion_matrix_postrun_{speed_tag.replace(" ", "").replace(".", "")}_bs{cfg["batch_size"]}_lr{cfg["lr"]}_wd{cfg["weight_decay"]}.png'
        _cm_save_path = os.path.join(_results_dir, _cm_fname)
        plt.savefig(_cm_save_path, dpi=150, bbox_inches='tight')
        print(f'[SAVED] Confusion matrix -> {_cm_save_path}')
        plt.show()

        del model
        torch.cuda.empty_cache()
        

In [ ]:
os.makedirs('results_resnet_full_evaluation', exist_ok=True)

#  SAVE BEST MODELS
import shutil 
for tag, best in [('1.83Hz', best_183), ('5.10Hz', best_510)]:
    dst = f'results_resnet_full_evaluation/rock_classifier_resnet18_{tag}.pth'
    shutil.copy(best['best_model_path'], dst)
    with open(f'results_resnet_full_evaluation/class_names_resnet18_{tag}.json', 'w') as f:
        json.dump(class_names, f)
    print(f'Saved {dst}')

print('\nTo view TensorBoard:')
print('  cd ~/results_resnet_full_evaluation/resnet18_results_183 && tensorboard --logdir=.')
print('  cd ~/results_resnet_full_evaluation/resnet18_results_510 && tensorboard --logdir=.')

summary = {
    '1.83Hz': {'best_acc': best_183['best_test_acc'], 'config': best_183['config']},
    '5.10Hz': {'best_acc': best_510['best_test_acc'], 'config': best_510['config']},
    'class_names': class_names
}
with open('results_resnet_full_evaluation/results_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

In [ ]:
# ARCHITECTURE COMPARISON 
# Runs ResNet-18/34/50 and EfficientNet-B0 on 1.83Hz with the best config.

_ARCH_LIST    = ['resnet18', 'resnet34', 'resnet50', 'efficientnet_b0']
_BEST_CONFIG  = best_183['config']
_ARCH_RESULTS = {}
_arch_dir     = 'results_resnet_full_evaluation/resnet18_results_arch_comparison'
os.makedirs(_arch_dir, exist_ok=True)

print(f'Reference config: {_BEST_CONFIG}\n')

for _arch in _ARCH_LIST:
    _sep = '=' * 35
    print(f'\n{_sep}\nArchitecture: {_arch}\n{_sep}')

    _model  = build_model(3, arch=_arch).to(device)
    _opt    = torch.optim.Adam(
        _model.parameters(),
        lr=_BEST_CONFIG['lr'],
        weight_decay=_BEST_CONFIG['weight_decay']
    )
    _sch    = torch.optim.lr_scheduler.ReduceLROnPlateau(_opt, mode='max', factor=0.5, patience=2)
    _crit   = nn.CrossEntropyLoss()
    _scaler = torch.amp.GradScaler(device.type if device.type == 'cuda' else 'cpu')
    _trl = DataLoader(train_dataset_183, batch_size=_BEST_CONFIG['batch_size'],
                      shuffle=True,  num_workers=4, pin_memory=True, persistent_workers=True)
    _tel = DataLoader(test_dataset_183,  batch_size=_BEST_CONFIG['batch_size'],
                      shuffle=False, num_workers=4, pin_memory=True, persistent_workers=True)

    _best_acc = 0.0
    for _e in range(10):
        _model.train()
        for _Xb, _yb in tqdm(_trl, leave=False):
            _Xb, _yb = _Xb.to(device), _yb.to(device)
            _opt.zero_grad()
            with torch.amp.autocast(device.type):
                _out  = _model(_Xb)
                _loss = _crit(_out, _yb)
            _scaler.scale(_loss).backward()
            _scaler.unscale_(_opt)
            torch.nn.utils.clip_grad_norm_(_model.parameters(), 0.5)
            _scaler.step(_opt)
            _scaler.update()

        _model.eval()
        _accs = []
        with torch.no_grad():
            for _Xb, _yb in _tel:
                _preds = torch.max(_model(_Xb.to(device)), 1)[1]
                _accs.append((_preds == _yb.to(device)).float().mean().item())
        _val = sum(_accs) / len(_accs)
        _sch.step(_val)
        if _val > _best_acc:
            _best_acc = _val
        print(f'  Epoch {_e+1}/10  val_acc={_val:.4f}')

    _ARCH_RESULTS[_arch] = _best_acc
    print(f'  Best: {_best_acc * 100:.2f}%')
    del _model, _opt, _sch, _scaler, _crit, _trl, _tel
    torch.cuda.empty_cache()

# Summary 
_sep2 = '=' * 50
print(f'\n{_sep2}')
print('ARCHITECTURE COMPARISON — 1.83 Hz  (10 epochs)')
print(_sep2)
for _a, _v in sorted(_ARCH_RESULTS.items(), key=lambda x: -x[1]):
    print(f'  {_a:<22}  {_v * 100:.2f}%')

# Bar chart
fig_ac, ax_ac = plt.subplots(figsize=(8, 5))
_names   = list(_ARCH_RESULTS.keys())
_vals    = [_ARCH_RESULTS[n] * 100 for n in _names]
_bars_ac = ax_ac.bar(
    _names, _vals,
    color=['#2196F3', '#4CAF50', '#FF9800', '#9C27B0'],
    edgecolor='black', width=0.5
)
ax_ac.bar_label(_bars_ac, fmt='%.2f%%', padding=3)
ax_ac.set_ylim(max(0, min(_vals) - 5), 100)
ax_ac.set_ylabel('Best Test Accuracy (%)')
ax_ac.set_title(
    f'Architecture Comparison — 1.83 Hz  '
    f'(10 epochs, {100-int(TEST_SPLIT*100)}/{int(TEST_SPLIT*100)} train/test split)'
)
ax_ac.grid(axis='y', alpha=0.3)
plt.tight_layout()
_ac_path = os.path.join(_arch_dir, 'architecture_comparison.png')
plt.savefig(_ac_path, dpi=150, bbox_inches='tight')
print(f'[SAVED] Architecture chart -> {_ac_path}')
plt.show()

with open(os.path.join(_arch_dir, 'arch_results.json'), 'w') as _f:
    json.dump({k: float(v) for k, v in _ARCH_RESULTS.items()}, _f, indent=2)
print('[SAVED] arch_results.json')


In [ ]:
# K-FOLD CROSS VALIDATION 
# 5-fold StratifiedKFold on the FULL dataset (before any train/test split). Gives mean accuracy ± std across folds — more robust than a single split.


def run_kfold(all_paths, all_labels, speed_tag, results_dir,
              n_classes, n_splits=5, kfold_epochs=10, arch='resnet18'):
    os.makedirs(results_dir, exist_ok=True)
    paths_arr  = np.array(all_paths)
    labels_arr = np.array(all_labels)
    kf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    fold_accs = []

    for fold, (tr_idx, val_idx) in enumerate(kf.split(paths_arr, labels_arr)):
        print(f'\n── Fold {fold+1}/{n_splits}  ({speed_tag}) ──')

        tr_ds  = SpectralImageDataset(
            paths_arr[tr_idx].tolist(), labels_arr[tr_idx].tolist(), transform=_active_train_transform)
        val_ds = SpectralImageDataset(
            paths_arr[val_idx].tolist(), labels_arr[val_idx].tolist(), transform=test_transform)

        tr_ld  = DataLoader(tr_ds,  batch_size=64, shuffle=True,
                            num_workers=4, pin_memory=True, persistent_workers=True)
        val_ld = DataLoader(val_ds, batch_size=64, shuffle=False,
                            num_workers=4, pin_memory=True, persistent_workers=True)

        model  = build_model(n_classes, arch=arch).to(device)
        opt    = torch.optim.Adam(model.parameters(), lr=1e-4)
        sch    = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=2)
        crit   = nn.CrossEntropyLoss()
        scaler = torch.amp.GradScaler(device.type if device.type == 'cuda' else 'cpu')

        best_fold_acc = 0.0
        best_true, best_pred = [], []

        for ep in range(kfold_epochs):
            model.train()
            for Xb, yb in tqdm(tr_ld, leave=False):
                Xb, yb = Xb.to(device), yb.to(device)
                opt.zero_grad()
                with torch.amp.autocast(device.type):
                    out  = model(Xb)
                    loss = crit(out, yb)
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
                scaler.step(opt)
                scaler.update()

            model.eval()
            ep_preds, ep_true, ep_accs = [], [], []
            with torch.no_grad():
                for Xb, yb in val_ld:
                    preds = torch.max(model(Xb.to(device)), 1)[1].cpu()
                    ep_preds.extend(preds.tolist())
                    ep_true.extend(yb.tolist())
                    ep_accs.append((preds == yb).float().mean().item())
            val_acc = sum(ep_accs) / len(ep_accs)
            sch.step(val_acc)
            if val_acc > best_fold_acc:
                best_fold_acc = val_acc
                best_true, best_pred = ep_true[:], ep_preds[:]
            print(f'  Epoch {ep+1}/{kfold_epochs}  val_acc={val_acc:.4f}')

        fold_accs.append(best_fold_acc)
        print(f'  Fold {fold+1} best: {best_fold_acc*100:.2f}%')
        print(classification_report(best_true, best_pred, target_names=class_names, digits=4))
        del model, opt, sch, scaler, crit, tr_ld, val_ld
        torch.cuda.empty_cache()

    # Summary 
    _sep = '=' * 50
    print(f'\n{_sep}')
    print(f'K-FOLD RESULTS — {speed_tag}  ({n_splits} folds, {arch})')
    print(_sep)
    for i, acc in enumerate(fold_accs):
        print(f'  Fold {i+1}: {acc*100:.2f}%')
    print(f'  Mean : {np.mean(fold_accs)*100:.2f}%')
    print(f'  Std  : {np.std(fold_accs)*100:.2f}%')

    # Bar chart
    fig_kf, ax_kf = plt.subplots(figsize=(7, 4))
    bars_kf = ax_kf.bar(range(1, n_splits+1), [a*100 for a in fold_accs],
                        color='steelblue', edgecolor='black')
    ax_kf.bar_label(bars_kf, fmt='%.2f%%', padding=3)
    ax_kf.axhline(np.mean(fold_accs)*100, color='red', linestyle='--',
                  label=f'Mean={np.mean(fold_accs)*100:.2f}% ± {np.std(fold_accs)*100:.2f}%')
    ax_kf.set_xlabel('Fold')
    ax_kf.set_ylabel('Best Val Accuracy (%)')
    ax_kf.set_title(f'K-fold CV — {speed_tag}  ({n_splits} folds, {arch})')
    ax_kf.legend()
    ax_kf.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    tag = speed_tag.replace(' ', '').replace('.', '')
    kf_path = os.path.join(results_dir, f'kfold_cv_{tag}.png')
    plt.savefig(kf_path, dpi=150, bbox_inches='tight')
    print(f'[SAVED] K-fold plot -> {kf_path}')
    plt.show()

    summary = {
        'fold_accs': fold_accs,
        'mean': float(np.mean(fold_accs)),
        'std':  float(np.std(fold_accs))
    }
    kf_json = os.path.join(results_dir, f'kfold_summary_{tag}.json')
    with open(kf_json, 'w') as _f:
        json.dump(summary, _f, indent=2)
    print(f'[SAVED] K-fold summary -> {kf_json}')
    return fold_accs


# Run on both speeds
kfold_183 = run_kfold(
    paths_183, labels_183, '1.83 Hz', 'results_resnet_full_evaluation/resnet18_results_183', n_classes=3, arch=ARCH)
kfold_510 = run_kfold(
    paths_510, labels_510, '5.10 Hz', 'results_resnet_full_evaluation/resnet18_results_510', n_classes=3, arch=ARCH)


In [ ]:
# GRAD-CAM VISUALIZATION 
# Gradient-weighted Class Activation Map(shows which regions of each spectral image drive the classification decision).

class GradCAM:
    """Computes Grad-CAM heatmaps via backward-hook on a target conv layer."""

    def __init__(self, model, target_layer):
        self.model       = model
        self.activations = None
        self.gradients   = None
        self._fwd_handle = target_layer.register_forward_hook(self._fwd_hook)
        self._bwd_handle = target_layer.register_full_backward_hook(self._bwd_hook)

    def remove_hooks(self):
        self._fwd_handle.remove()
        self._bwd_handle.remove()

    def _fwd_hook(self, m, inp, out):
        self.activations = out.detach()

    def _bwd_hook(self, m, gin, gout):
        self.gradients = gout[0].detach()

    def generate(self, x, class_idx=None):
        """Returns (cam_normalized 0-1, predicted_class_idx)."""
        self.model.eval()
        self.model.zero_grad()
        out = self.model(x)
        pred_class = out.argmax(1).item() 
        if class_idx is None:
            class_idx = pred_class
        out[0, class_idx].backward()
        weights = self.gradients.mean(dim=[2, 3], keepdim=True)  
        cam = F.relu((weights * self.activations).sum(dim=1, keepdim=True))
        cam = F.interpolate(cam, size=(224, 224), mode='bilinear', align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, pred_class 


def _gradcam_target_layer(model, arch):
    """Return the last convolutional block for Grad-CAM."""
    if arch in ('resnet18', 'resnet34', 'resnet50'):
        return model.layer4[-1]
    elif arch == 'efficientnet_b0':
        return model.features[-1]
    raise ValueError(f'No Grad-CAM target layer defined for arch={arch}')


def visualize_gradcam(model_path, test_dataset, class_names, speed_tag,
                      results_dir, arch='resnet18', n_samples=3):
    """Generate Grad-CAM overlays for n_samples per class and save to disk."""
    model = build_model(len(class_names), arch=arch).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
    model.eval()

    gc = GradCAM(model, _gradcam_target_layer(model, arch))

    
    inv_norm = transforms.Normalize(
        mean=[-m/s for m, s in zip([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])],
        std =[1/s   for s    in                             [0.229, 0.224, 0.225]]
    )

    n_cls = len(class_names)
    fig, axes = plt.subplots(n_cls, n_samples * 2, figsize=(n_samples * 5, n_cls * 3.5))
    fig.suptitle(f'Grad-CAM — {speed_tag}  |  {arch}', fontsize=13, fontweight='bold')

    for ci, cls_name in enumerate(class_names):
        idxs = [i for i, lbl in enumerate(test_dataset.labels) if lbl == ci][:n_samples]

        for j, idx in enumerate(idxs):
            img_t, _ = test_dataset[idx]
            x = img_t.unsqueeze(0).to(device)
            cam, pred_ci = gc.generate(x, class_idx=ci)
            img_np = inv_norm(img_t).permute(1, 2, 0).clamp(0, 1).cpu().numpy()

            correct = chr(10003) if pred_ci == ci else chr(10007)   

            ax_o = axes[ci][j * 2]
            ax_c = axes[ci][j * 2 + 1]

            ax_o.imshow(img_np)
            ax_o.set_title(cls_name, fontsize=8, fontweight='bold')
            ax_o.axis('off')

            ax_c.imshow(img_np)
            ax_c.imshow(cam, alpha=0.45, cmap='jet')
            ax_c.set_title(f'pred={class_names[pred_ci]} {correct}', fontsize=7)
            ax_c.axis('off')

    plt.tight_layout()
    os.makedirs(results_dir, exist_ok=True)
    tag = speed_tag.replace(' ', '').replace('.', '')
    sp  = os.path.join(results_dir, f'gradcam_{tag}.png')
    plt.savefig(sp, dpi=150, bbox_inches='tight')
    print(f'[SAVED] Grad-CAM -> {sp}')
    plt.show()
    gc.remove_hooks()
    del model
    torch.cuda.empty_cache()


# Run on best checkpoint from each speed 
print('Generating Grad-CAM for 1.83 Hz ...')
visualize_gradcam(
    best_183['best_model_path'], test_dataset_183,
    class_names, '1.83 Hz', 'results_resnet_full_evaluation/resnet18_results_183', arch=ARCH, n_samples=3
)

print('\nGenerating Grad-CAM for 5.10 Hz ...')
visualize_gradcam(
    best_510['best_model_path'], test_dataset_510,
    class_names, '5.10 Hz', 'results_resnet_full_evaluation/resnet18_results_510', arch=ARCH, n_samples=3
)


In [ ]:
# MISCLASSIFICATION ANALYSIS + GRAD-CAM 
# Finds images the best model got wrong, shows them with Grad-CAM overlay.

def misclassification_analysis(model_path, test_dataset, class_names, speed_tag,
                                results_dir, arch='resnet18', n_per_pair=2):
    model = build_model(len(class_names), arch=arch).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
    model.eval()
    gc = GradCAM(model, _gradcam_target_layer(model, arch))

    inv_norm = transforms.Normalize(
        mean=[-m/s for m, s in zip([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])],
        std =[1/s   for s    in                             [0.229, 0.224, 0.225]]
    )

    wrong = {(t, p): [] for t in range(len(class_names))
                         for p in range(len(class_names)) if t != p}

    loader = DataLoader(test_dataset, batch_size=64, shuffle=False,
                        num_workers=4, pin_memory=True, persistent_workers=True)
    idx_offset = 0
    with torch.no_grad():
        for Xb, yb in loader:
            preds = torch.max(model(Xb.to(device)), 1)[1].cpu()
            for k in range(len(yb)):
                t, p = int(yb[k]), int(preds[k])
                if t != p:
                    wrong[(t, p)].append(idx_offset + k)
            idx_offset += len(yb)

    # Filter pairs that actually have errors
    pairs = [(t, p) for (t, p), idxs in wrong.items() if len(idxs) > 0]
    if not pairs:
        print(f'No misclassifications found for {speed_tag}!')
        gc.remove_hooks()
        del model
        return

    n_pairs = len(pairs)
    fig, axes = plt.subplots(n_pairs, n_per_pair * 2,
                             figsize=(n_per_pair * 5, n_pairs * 3.5),
                             squeeze=False)
    fig.suptitle(f'Misclassification Analysis + Grad-CAM — {speed_tag}',
                 fontsize=12, fontweight='bold')

    for row, (t, p) in enumerate(pairs):
        sample_idxs = wrong[(t, p)][:n_per_pair]
        for col, ds_idx in enumerate(sample_idxs):
            img_t, _ = test_dataset[ds_idx]
            x        = img_t.unsqueeze(0).to(device)
            cam, _   = gc.generate(x, class_idx=p)   
            img_np   = inv_norm(img_t).permute(1, 2, 0).clamp(0, 1).cpu().numpy()

            ax_o = axes[row][col * 2]
            ax_c = axes[row][col * 2 + 1]

            ax_o.imshow(img_np)
            ax_o.set_title(f'TRUE: {class_names[t]}', fontsize=7, color='green')
            ax_o.axis('off')

            ax_c.imshow(img_np)
            ax_c.imshow(cam, alpha=0.45, cmap='jet')
            ax_c.set_title(f'PRED: {class_names[p]}', fontsize=7, color='red')
            ax_c.axis('off')

    plt.tight_layout()
    os.makedirs(results_dir, exist_ok=True)
    tag  = speed_tag.replace(' ', '').replace('.', '')
    path = os.path.join(results_dir, f'misclassification_gradcam_{tag}.png')
    plt.savefig(path, dpi=150, bbox_inches='tight')
    print(f'[SAVED] Misclassification analysis -> {path}')
    plt.show()

    # Summary table
    print(f'\nMisclassification summary — {speed_tag}:')
    for (t, p) in sorted(pairs):
        print(f'  {class_names[t]:<25} misclassified as {class_names[p]:<25}: {len(wrong[(t,p)])} images')

    gc.remove_hooks()
    del model
    torch.cuda.empty_cache()


print('Misclassification analysis — 1.83 Hz ...')
misclassification_analysis(
    best_183['best_model_path'], test_dataset_183,
    class_names, '1.83 Hz', 'results_resnet_full_evaluation/resnet18_results_183', arch=ARCH)

print('\nMisclassification analysis — 5.10 Hz ...')
misclassification_analysis(
    best_510['best_model_path'], test_dataset_510,
    class_names, '5.10 Hz', 'results_resnet_full_evaluation/resnet18_results_510', arch=ARCH)


In [ ]:
# CROSS-SPEED GRAD-CAM COMPARISON 
# Shows the saem rock class side-by-side: 1.83 Hz image vs 5.10 Hz image.

def gradcam_speed_comparison(model_path_183, model_path_510,
                              test_ds_183, test_ds_510,
                              class_names, results_dir, arch='resnet18', n_samples=2):
    inv_norm = transforms.Normalize(
        mean=[-m/s for m, s in zip([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])],
        std =[1/s   for s    in                             [0.229, 0.224, 0.225]]
    )

    models_info = [
        ('1.83 Hz', model_path_183, test_ds_183),
        ('5.10 Hz', model_path_510, test_ds_510),
    ]

    n_cls  = len(class_names)
    n_cols = n_samples * 2 * 2   
    fig, axes = plt.subplots(n_cls, n_cols,
                             figsize=(n_cols * 2.5, n_cls * 3.5),
                             squeeze=False)
    fig.suptitle(f'Cross-Speed Grad-CAM Comparison  ({arch})',
                 fontsize=12, fontweight='bold')

    for ci, cls_name in enumerate(class_names):
        col = 0
        for speed_tag, model_path, test_ds in models_info:
            model = build_model(len(class_names), arch=arch).to(device)
            model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
            model.eval()
            gc = GradCAM(model, _gradcam_target_layer(model, arch))

            idxs = [i for i, lbl in enumerate(test_ds.labels) if lbl == ci][:n_samples]
            for idx in idxs:
                img_t, _ = test_ds[idx]
                x        = img_t.unsqueeze(0).to(device)
                cam, pred_ci = gc.generate(x, class_idx=ci)
                img_np   = inv_norm(img_t).permute(1, 2, 0).clamp(0, 1).cpu().numpy()
                correct  = chr(10003) if pred_ci == ci else chr(10007)

                axes[ci][col].imshow(img_np)
                axes[ci][col].set_title(f'{cls_name}\n{speed_tag}', fontsize=6)
                axes[ci][col].axis('off')
                col += 1

                axes[ci][col].imshow(img_np)
                axes[ci][col].imshow(cam, alpha=0.45, cmap='jet')
                axes[ci][col].set_title(f'CAM {correct}', fontsize=6)
                axes[ci][col].axis('off')
                col += 1

            gc.remove_hooks()
            del model
            torch.cuda.empty_cache()

    plt.tight_layout()
    os.makedirs(results_dir, exist_ok=True)
    path = os.path.join(results_dir, 'gradcam_speed_comparison.png')
    plt.savefig(path, dpi=150, bbox_inches='tight')
    print(f'[SAVED] Cross-speed Grad-CAM -> {path}')
    plt.show()


gradcam_speed_comparison(
    best_183['best_model_path'], best_510['best_model_path'],
    test_dataset_183, test_dataset_510,
    class_names, 'results', arch=ARCH, n_samples=2)


In [ ]:
# t-SNE FEATURE VISUALIZATION 
# Extracts the penultimate feature vector (just before the fc layer) for every test image, then reduces to 2D with t-SNE.

def extract_features(model_path, dataset, n_classes, arch='resnet18'):
    """Return (features [N, D], labels [N]) numpy arrays. D=512 for ResNet, 1280 for EfficientNet."""
    model = build_model(n_classes, arch=arch).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
    model.eval()

    features_list = []
    def _hook(m, inp, out):
        features_list.append(out.detach().cpu())

    if 'resnet' in arch:
        handle = model.avgpool.register_forward_hook(_hook)
    else: 
        handle = model.avgpool.register_forward_hook(_hook)

    loader = DataLoader(dataset, batch_size=64, shuffle=False,
                        num_workers=4, pin_memory=True, persistent_workers=True)
    all_labels = []
    with torch.no_grad():
        for Xb, yb in tqdm(loader, desc='Extracting features', leave=False):
            model(Xb.to(device))
            all_labels.extend(yb.tolist())

    handle.remove()
    del model
    torch.cuda.empty_cache()

    feats = torch.cat(features_list, dim=0).reshape(len(all_labels), -1).numpy()  
    return feats, np.array(all_labels)


def plot_tsne(feats, labels, class_names, speed_tag, results_dir, perplexity=30):
    from sklearn.manifold import TSNE
    print(f'  Running t-SNE on {feats.shape[0]} samples ...')
    tsne   = TSNE(n_components=2, perplexity=perplexity, random_state=42, max_iter=1000)
    emb    = tsne.fit_transform(feats)

    colors = ['#E53935', '#1E88E5', '#43A047']
    fig_ts, ax_ts = plt.subplots(figsize=(8, 6))
    for ci, cls_name in enumerate(class_names):
        mask = labels == ci
        ax_ts.scatter(emb[mask, 0], emb[mask, 1],
                      c=colors[ci], label=cls_name,
                      alpha=0.5, s=12, edgecolors='none')
    ax_ts.set_title(f't-SNE Feature Space — {speed_tag}  (perplexity={perplexity})')
    ax_ts.legend(markerscale=2)
    ax_ts.set_xlabel('t-SNE dim 1')
    ax_ts.set_ylabel('t-SNE dim 2')
    ax_ts.grid(True, alpha=0.2)
    plt.tight_layout()
    os.makedirs(results_dir, exist_ok=True)
    tag  = speed_tag.replace(' ', '').replace('.', '')
    path = os.path.join(results_dir, f'tsne_{tag}.png')
    plt.savefig(path, dpi=150, bbox_inches='tight')
    print(f'[SAVED] t-SNE -> {path}')
    plt.show()


print('t-SNE — 1.83 Hz ...')
feats_183, lbls_183 = extract_features(
    best_183['best_model_path'], test_dataset_183, n_classes=3, arch=ARCH)
plot_tsne(feats_183, lbls_183, class_names, '1.83 Hz',
          'results_resnet_full_evaluation/resnet18_results_183')

print('\nt-SNE — 5.10 Hz ...')
feats_510, lbls_510 = extract_features(
    best_510['best_model_path'], test_dataset_510, n_classes=3, arch=ARCH)
plot_tsne(feats_510, lbls_510, class_names, '5.10 Hz',
          'results_resnet_full_evaluation/resnet18_results_510')


In [ ]:
#  TEST-TIME AUGMENTATION (TTA) 
# During inference, each image is augmented N times and predictions are averaged (makes the model more robust to slight variations in belt speed / lighting).

tta_transforms = [
    transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
    ]),
    transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.ToTensor(),
        transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
    ]),
    transforms.Compose([
        transforms.Resize((236, 236)),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
    ]),
    transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ColorJitter(brightness=0.1, contrast=0.1),
        transforms.ToTensor(),
        transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
    ]),
    transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomRotation(degrees=5),
        transforms.ToTensor(),
        transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
    ]),
]


def evaluate_with_tta(model_path, image_paths, labels, class_names,
                       speed_tag, results_dir, arch='resnet18'):
    model = build_model(len(class_names), arch=arch).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
    model.eval()

    all_true, all_pred_tta, all_pred_base = [], [], []

    with torch.no_grad():
        for img_path, true_label in tqdm(
                zip(image_paths, labels), total=len(labels),
                desc=f'TTA {speed_tag}', leave=False):
            img = Image.open(img_path).convert('RGB')

            x_base  = test_transform(img).unsqueeze(0).to(device)
            logits_base = model(x_base)
            pred_base   = int(logits_base.argmax(1).item())

            probs = torch.zeros(1, len(class_names)).to(device)
            for t in tta_transforms:
                x   = t(img).unsqueeze(0).to(device)
                probs += torch.softmax(model(x), dim=1)
            probs /= len(tta_transforms)
            pred_tta = int(probs.argmax(1).item())

            all_true.append(true_label)
            all_pred_base.append(pred_base)
            all_pred_tta.append(pred_tta)

    base_acc = sum(p == t for p, t in zip(all_pred_base, all_true)) / len(all_true)
    tta_acc  = sum(p == t for p, t in zip(all_pred_tta,  all_true)) / len(all_true)

    print(f'\nTTA Results — {speed_tag}:')
    print(f'  Baseline accuracy : {base_acc*100:.2f}%')
    print(f'  TTA accuracy      : {tta_acc*100:.2f}%  (delta: {(tta_acc-base_acc)*100:+.2f}%)')
    print(f'\nTTA Classification Report:\n')
    print(classification_report(all_true, all_pred_tta,
                                target_names=class_names, digits=4))

    # Save confusion matrix
    cm   = confusion_matrix(all_true, all_pred_tta)
    fig_tta, ax_tta = plt.subplots(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=ax_tta)
    ax_tta.set_title(f'TTA Confusion Matrix — {speed_tag}\n'
                     f'Base: {base_acc*100:.2f}%  TTA: {tta_acc*100:.2f}%')
    ax_tta.set_ylabel('Actual')
    ax_tta.set_xlabel('Predicted')
    plt.tight_layout()
    os.makedirs(results_dir, exist_ok=True)
    tag  = speed_tag.replace(' ', '').replace('.', '')
    path = os.path.join(results_dir, f'tta_confusion_matrix_{tag}.png')
    plt.savefig(path, dpi=150, bbox_inches='tight')
    print(f'[SAVED] TTA CM -> {path}')
    plt.show()

    del model
    torch.cuda.empty_cache()
    return base_acc, tta_acc


tta_base_183, tta_acc_183 = evaluate_with_tta(
    best_183['best_model_path'], paths_test_183, labels_test_183,
    class_names, '1.83 Hz', 'results_resnet_full_evaluation/resnet18_results_183', arch=ARCH)

tta_base_510, tta_acc_510 = evaluate_with_tta(
    best_510['best_model_path'], paths_test_510, labels_test_510,
    class_names, '5.10 Hz', 'results_resnet_full_evaluation/resnet18_results_510', arch=ARCH)

print(f'''
TTA SUMMARY
  1.83 Hz: {tta_base_183*100:.2f}% -> {tta_acc_183*100:.2f}%  ({(tta_acc_183-tta_base_183)*100:+.2f}%)
  5.10 Hz: {tta_base_510*100:.2f}% -> {tta_acc_510*100:.2f}%  ({(tta_acc_510-tta_base_510)*100:+.2f}%)
''')
